# 01_prepare_data

## 목적
- 문제를 **과거 이력으로 다음 주문의 지연 위험을 예측**하는 구조로 고정한다.
- `feature`와 `label`의 시점을 분리해 데이터 누수를 막는다.
- high-frequency threshold = `24`, delay threshold = `15`의 근거를 문서화한다.

## 입력 파일
- `model_base_hv.csv`: 현재 저장소의 기준 전처리 산출물

## 출력 파일
- 권장: `results/train_ids.csv`, `results/val_ids.csv`, `results/test_ids.csv`
- 권장: `results/preprocessing_report.md`

## 핵심 설명
1. `target` 주문은 label 생성용으로만 사용한다.
2. `feature`는 반드시 target 주문 **이전 이력만** 사용한다.
3. 현재 CSV 기준으로 high-frequency threshold는 `24`로 읽힌다.
4. 라벨은 `target_gap > 15 -> delay_risk = 1`로 정의한다.
5. split은 사용자 단위 stratified `train / val / test = 70 / 15 / 15`를 권장한다.

## NOTE
- 발표 기획에서는 `15일`을 `q90 ≈ 15일` 근거로 설명하고자 한다.
- 하지만 현재 `model_base_hv.csv` 재계산 결과는 `q80 = 15`, `q90 = 24`다.
- 따라서 발표에서는 초기 기획 근거와 현재 CSV 재계산값을 분리해서 설명해야 한다.


In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("model_base_hv.csv")
df = pd.read_csv(DATA_PATH)

summary = {
    "rows": len(df),
    "positive_ratio": round(df["delay_risk"].mean(), 6),
    "min_target_order_number": int(df["target_order_number"].min()),
    "max_target_order_number": int(df["target_order_number"].max()),
    "q80_target_gap": float(df["target_gap"].quantile(0.80)),
    "q90_target_gap": float(df["target_gap"].quantile(0.90)),
}

summary
